# HalluGuard × MiniCheck — full benchmark

Runs all 29,320 LLM-AggreFact test rows through `MiniCheck.score()`. HalluGuard uses deterministic no-thinking exact `YES`/`NO`; failures are recorded without guessed labels.


## 1. Setup


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WORKING = Path("/kaggle/working")
if not WORKING.exists():
    WORKING = Path.cwd()
SOURCE_DIR = WORKING / "minicheck_halluguard_binary_source"
MINICHECK_COMMIT = "b58b9fa69acbd1015ec970fa65dd752413a053d2"

def run_checked(args, cwd=None):
    print("+", " ".join(map(str, args)))
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True, text=True)

gpu_before = subprocess.run(
    ["nvidia-smi"], text=True, capture_output=True, check=True
).stdout
print(gpu_before)
assert "T4" in gpu_before, "This notebook targets Kaggle T4 GPUs."

if SOURCE_DIR.exists():
    shutil.rmtree(SOURCE_DIR)
run_checked([
    "git", "clone", "--filter=blob:none", "https://github.com/Liyan06/MiniCheck.git", str(SOURCE_DIR)
])
run_checked(["git", "checkout", MINICHECK_COMMIT], cwd=SOURCE_DIR)

run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "vllm==0.10.2", "transformers==4.57.6", "datasets", "nltk", "scikit-learn",
    "pandas", "huggingface_hub", "accelerate"
])

# Read-only credential handling. Never place token values in notebook source or output.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN is available from the environment/Kaggle Secrets (value not printed).")
else:
    print("HF_TOKEN is not set; continuing because the referenced public assets may not require it.")


## 2. MiniCheck binary HalluGuard adapter


In [ ]:
from pathlib import Path

mini_path = SOURCE_DIR / "minicheck" / "minicheck.py"
inf_path = SOURCE_DIR / "minicheck" / "inference.py"
mini = mini_path.read_text(encoding="utf-8")
inf = inf_path.read_text(encoding="utf-8")

allowed_tail = "'Granite-Guardian-3.3-8B']"
assert mini.count(allowed_tail) == 2, "Pinned minicheck.py no longer matches the reviewed source."
mini = mini.replace(
    allowed_tail,
    "'Granite-Guardian-3.3-8B', 'HalluGuard-Qwen3-4B']",
    2,
)

anchor = """        elif model_name == 'Granite-Guardian-3.3-8B':
            if not max_tokens or max_tokens<2048:
                print(\"For Granite Guardian 3.3 - fixing the max_tokens to be 2048\")
                max_tokens=2048

            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )"""
replacement = anchor + """
        elif model_name == 'HalluGuard-Qwen3-4B':
            if max_tokens == 1:
                print(\"HalluGuard reasoning requires more than one token; using max_tokens=2048.\")
                max_tokens = 2048
            if not max_tokens or max_tokens < 2:
                raise ValueError(\"HalluGuard max_tokens must be at least 2.\")
            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )"""
assert anchor in mini, "Could not locate the MiniCheck model-construction anchor."
mini = mini.replace(anchor, replacement, 1)

inf = inf.replace("import random\n", "import random\nimport json\nimport re\n", 1)

anchor = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        else:
            raise ValueError(\"model_id must be 'Bespoke-MiniCheck-7B'\")"""
replacement = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        elif model_id == 'HalluGuard-Qwen3-4B':
            self.model_id = 'lrsbrgrn/HalluGuard-Qwen3-4B'
            self.operating_mode = \"halluguard\"
            self.model_revision = '2bf3858f8bd0c22b0c410f6b0c855098727a2554'
            self.vllm_kwargs = {
                'enforce_eager': True,
                'disable_custom_all_reduce': True,
                'gpu_memory_utilization': 0.80,
                'max_num_seqs': 16,
            }
        else:
            raise ValueError(\"Unsupported LLMCheck model_id: %s\" % model_id)"""
assert anchor in inf, "Could not locate the LLMCheck model-id anchor."
inf = inf.replace(anchor, replacement, 1)

anchor = "        if model_id == 'Bespoke-MiniCheck-7B':\n"
replacement = "        self.model_revision = None\n        self.vllm_kwargs = {}\n\n" + anchor
assert anchor in inf, "Could not locate model revision insertion anchor."
inf = inf.replace(anchor, replacement, 1)

anchor = "            model=self.model_id, "
replacement = "            model=self.model_id,\n            revision=self.model_revision, "
assert anchor in inf, "Could not locate vLLM model construction."
inf = inf.replace(anchor, replacement, 1)

# Kaggle dual-T4 stability: HalluGuard alone uses eager mode and NCCL fallback.
# This avoids the observed CUDA-graph/custom-all-reduce crash without changing
# the runtime defaults of Bespoke MiniCheck or Granite Guardian.
anchor = "            enable_prefix_caching=self.enable_prefix_caching"
replacement = "            enable_prefix_caching=self.enable_prefix_caching,\n            **self.vllm_kwargs"
assert inf.count(anchor) == 1, "Could not locate vLLM stability kwargs insertion point."
inf = inf.replace(anchor, replacement, 1)

anchor = """        self.sampling_params = SamplingParams(
            temperature=0,
            max_tokens=self.max_tokens,
            stop_token_ids=terminators,
            logprobs=5
        )"""
replacement = """        if self.operating_mode == \"halluguard\":
            # Official HalluGuard/Qwen3 reasoning settings, bounded by this experiment's
            # frozen 4,096-token context. A truncated answer is a parse failure, never a guess.
            self.sampling_params = SamplingParams(
                temperature=0,
                top_p=0.95,
                top_k=20,
                max_tokens=self.max_tokens,
                stop_token_ids=terminators,
                seed=42
            )
        else:
            self.sampling_params = SamplingParams(
                temperature=0,
                max_tokens=self.max_tokens,
                stop_token_ids=terminators,
                logprobs=5
            )
        self.last_halluguard_records = []"""
assert anchor in inf, "Could not locate SamplingParams construction."
inf = inf.replace(anchor, replacement, 1)

anchor = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        return text"""
replacement = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        elif self.operating_mode == \"halluguard\":
            halluguard_prompt = json.dumps({
                \"instructions\": [
                    \"You will be given a document and a claim.\",
                    \"Decide whether the claim is 'GROUNDED', 'HALLUCINATED_INTRINSIC', or 'HALLUCINATED_EXTRINSIC' based ONLY on the document.\",
                    \"Definitions:\",
                    \"  - GROUNDED: The claim is fully supported by the document. All relevant parts are directly verifiable from the document.\",
                    \"  - HALLUCINATED_INTRINSIC: The claim contradicts what the document states or clearly implies.\",
                    \"  - HALLUCINATED_EXTRINSIC: The claim includes information that is not stated or implied in the document and cannot be verified using only the document (it requires external knowledge).\",
                    \"Justification requirements:\",
                    \"  - Your justification MUST be evidence-grounded.\",
                    \"  - Explicitly refer to the relevant parts of the document (by quoting or paraphrasing them).\",
                    \"  - Explain how these parts SUPPORT, CONTRADICT, or FAIL TO SUPPORT the claim.\",
                    \"  - Do NOT use any external knowledge; rely only on the provided document.\",
                    \"Answer format (VERY IMPORTANT):\",
                    \"  - You MUST respond using EXACTLY the following XML structure:\",
                    \"    <answer>\",
                    \"      <classification>CATEGORY</classification>\",
                    \"      <justification>Your reasoning here</justification>\",
                    \"    </answer>\",
                    \"  - CATEGORY must be ONE of: GROUNDED, HALLUCINATED_INTRINSIC, HALLUCINATED_EXTRINSIC.\",
                    \"  - The <justification> must briefly explain your reasoning and cite evidence from the document.\",
                    \"  - Do NOT add any other text before or after the <answer>...</answer> block.\",
                    \"  - Do NOT add any extra tags or attributes.\"
                ],
                \"document\": \"'%s'\" % doc,
                \"claim\": \"'%s'\" % claim
            })
            text = self.tokenizer.apply_chat_template(
                [{\"role\": \"user\", \"content\": halluguard_prompt}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            )
        return text"""
if anchor in inf:
    inf = inf.replace(anchor, replacement, 1)
else:
    print("Skipping brittle HalluGuard chat-template anchor; direct binary override will be installed.")

anchor = '    def get_support_prob_hybrid_gg(self, response, marker="score"):\n'
insert = """    def parse_halluguard_output(self, response_text):
        allowed = {
            \"GROUNDED\": 1.0,
            \"HALLUCINATED_INTRINSIC\": 0.0,
            \"HALLUCINATED_EXTRINSIC\": 0.0,
        }
        final_text = response_text.rsplit(\"</think>\", 1)[-1].strip()
        pattern = re.compile(
            r\"<answer>\\s*<classification>\\s*(GROUNDED|HALLUCINATED_INTRINSIC|HALLUCINATED_EXTRINSIC)\\s*</classification>\\s*\"
            r\"<justification>([\\s\\S]+?)</justification>\\s*</answer>\"
        )
        match = pattern.fullmatch(final_text)
        if match is None or not match.group(2).strip():
            raise HalluGuardParseError(
                \"HalluGuard output did not exactly match the required answer/classification/justification XML. \"
                + \"Raw output: \" + repr(response_text[:2000])
            )
        label = match.group(1)
        return label, allowed[label], final_text

    def get_support_score_halluguard(self, response):
        label, score, final_text = self.parse_halluguard_output(response.outputs[0].text)
        self.last_halluguard_records.append({
            \"classification\": label,
            \"support_score\": score,
            \"final_text\": final_text,
        })
        return score

""" + anchor
assert anchor in inf, "Could not locate HalluGuard parser insertion anchor."
inf = inf.replace(anchor, insert, 1)

inf = inf.replace(
    "class LLMCheck:\n",
    "class HalluGuardParseError(ValueError):\n    pass\n\n\nclass LLMCheck:\n",
    1,
)

anchor = """        if self.operating_mode==\"bespoke\":
            probs_per_chunk_sentence = [self.get_support_prob(responses[idx]) for idx in range(len(responses))]
        elif self.operating_mode==\"gg_hybrid\":
            probs_per_chunk_sentence = [self.get_support_prob_hybrid_gg(responses[idx]) for idx in range(len(responses))]"""
replacement = """        if self.operating_mode==\"bespoke\":
            probs_per_chunk_sentence = [self.get_support_prob(responses[idx]) for idx in range(len(responses))]
        elif self.operating_mode==\"gg_hybrid\":
            probs_per_chunk_sentence = [self.get_support_prob_hybrid_gg(responses[idx]) for idx in range(len(responses))]
        elif self.operating_mode == \"halluguard\":
            self.last_halluguard_records = []
            probs_per_chunk_sentence = [self.get_support_score_halluguard(response) for response in responses]"""
assert anchor in inf, "Could not locate response parsing dispatch."
inf = inf.replace(anchor, replacement, 1)


# Full-benchmark override: no thinking/XML; only exact YES or NO is accepted.
import re
inf = inf.replace('self.operating_mode = "halluguard"', 'self.operating_mode = "halluguard_binary"')
inf = re.sub(r'        if self\.operating_mode == "halluguard_binary":\n.*?        else:\n            self\.sampling_params = SamplingParams\(', """        if self.operating_mode == "halluguard_binary":
            self.sampling_params = SamplingParams(temperature=0, max_tokens=self.max_tokens, stop_token_ids=terminators, seed=42)
        else:
            self.sampling_params = SamplingParams(""", inf, count=1, flags=re.S)
inf = re.sub(r'        elif self\.operating_mode == "halluguard_binary":\n.*?        return text', """        elif self.operating_mode == "halluguard_binary":
            prompt = "Document:\\n" + doc + "\\n\\nClaim:\\n" + claim + "\\n\\nReply exactly YES if the claim is fully supported by the document. Reply exactly NO if the claim is contradicted or unsupported by the document. No explanation, reasoning, punctuation, XML, or extra text."
            text = self.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return text""", inf, count=1, flags=re.S)
inf = re.sub(r'    def parse_halluguard_output\(self, response_text\):\n.*?    def get_support_prob_hybrid_gg', """    def parse_halluguard_output(self, response_text):
        final_text = response_text.strip()
        if final_text == "YES": return "GROUNDED", 1.0, final_text
        if final_text == "NO": return "HALLUCINATED", 0.0, final_text
        raise HalluGuardParseError("HalluGuard binary output must be exactly trimmed uppercase YES or NO. Raw output: " + repr(response_text[:4000]))
    def get_support_score_halluguard(self, response):
        raw_text = response.outputs[0].text
        label, score, final_text = self.parse_halluguard_output(raw_text)
        self.last_halluguard_records.append({"classification": label, "support_score": score, "raw_output": raw_text})
        return score
    def get_support_prob_hybrid_gg""", inf, count=1, flags=re.S)
inf = inf.replace('elif self.operating_mode == "halluguard":\n            self.last_halluguard_records = []', 'elif self.operating_mode == "halluguard_binary":\n            self.last_halluguard_records = []')


# Full-benchmark override: no thinking/XML; only exact YES or NO is accepted.
import re
inf = inf.replace('self.operating_mode = "halluguard"', 'self.operating_mode = "halluguard_binary"')
inf = re.sub(r'        if self\.operating_mode == "halluguard_binary":\n.*?        else:\n            self\.sampling_params = SamplingParams\(', """        if self.operating_mode == "halluguard_binary":
            self.sampling_params = SamplingParams(temperature=0, max_tokens=self.max_tokens, stop_token_ids=terminators, seed=42)
        else:
            self.sampling_params = SamplingParams(""", inf, count=1, flags=re.S)
inf = re.sub(r'        elif self\.operating_mode == "halluguard_binary":\n.*?        return text', """        elif self.operating_mode == "halluguard_binary":
            prompt = "Document:\\n" + doc + "\\n\\nClaim:\\n" + claim + "\\n\\nReply exactly YES if the claim is fully supported by the document. Reply exactly NO if the claim is contradicted or unsupported by the document. No explanation, reasoning, punctuation, XML, or extra text."
            text = self.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return text""", inf, count=1, flags=re.S)
inf = re.sub(r'    def parse_halluguard_output\(self, response_text\):\n.*?    def get_support_prob_hybrid_gg', """    def parse_halluguard_output(self, response_text):
        final_text = response_text.strip()
        if final_text == "YES": return "GROUNDED", 1.0, final_text
        if final_text == "NO": return "HALLUCINATED", 0.0, final_text
        raise HalluGuardParseError("HalluGuard binary output must be exactly trimmed uppercase YES or NO. Raw output: " + repr(response_text[:4000]))
    def get_support_score_halluguard(self, response):
        raw_text = response.outputs[0].text
        label, score, final_text = self.parse_halluguard_output(raw_text)
        self.last_halluguard_records.append({"classification": label, "support_score": score, "raw_output": raw_text})
        return score
    def get_support_prob_hybrid_gg""", inf, count=1, flags=re.S)
inf = inf.replace('elif self.operating_mode == "halluguard":\n            self.last_halluguard_records = []', 'elif self.operating_mode == "halluguard_binary":\n            self.last_halluguard_records = []')


# Stable direct override: works even if the upstream apply_chat_template layout changes.
inf += '''\n\n_original_apply_chat_template = LLMCheck.apply_chat_template\ndef _halluguard_binary_apply_chat_template(self, doc, claim):\n    if self.operating_mode == "halluguard_binary":\n        prompt = "Document:\\n" + doc + "\\n\\nClaim:\\n" + claim + "\\n\\nReply exactly YES if the claim is fully supported by the document. Reply exactly NO if the claim is contradicted or unsupported by the document. No explanation, reasoning, punctuation, XML, or extra text."\n        return self.tokenizer.apply_chat_template([{\"role\": \"user\", \"content\": prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    return _original_apply_chat_template(self, doc, claim)\nLLMCheck.apply_chat_template = _halluguard_binary_apply_chat_template\n'''

mini_path.write_text(mini, encoding="utf-8")
inf_path.write_text(inf, encoding="utf-8")
print("Patched MiniCheck with the HalluGuard LLMCheck operating mode.")

run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", str(SOURCE_DIR)])

import nltk
for resource in ("punkt", "punkt_tab"):
    nltk.download(resource, quiet=True)

# Fresh import from the patched, pinned checkout.
sys.path.insert(0, str(SOURCE_DIR))
from minicheck.minicheck import MiniCheck
from minicheck.inference import HalluGuardParseError
print("MiniCheck import path:", __import__("minicheck").__file__)


In [ ]:
import json
import platform
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import balanced_accuracy_score

MODEL_NAME = "HalluGuard-Qwen3-4B"
MODEL_ID = "lrsbrgrn/HalluGuard-Qwen3-4B"
MODEL_REVISION = "2bf3858f8bd0c22b0c410f6b0c855098727a2554"
DATASET_ID = "lytang/LLM-AggreFact"
SPLIT = "test"
CHUNK_SIZE = 500
MAX_MODEL_LEN = 4096
MAX_TOKENS = 8
TENSOR_PARALLEL_SIZE = 1
ENABLE_PREFIX_CACHING = False
RANDOM_STATE = 42
CACHE_DIR = str(WORKING / "hf_cache")

assert torch.cuda.is_available() and torch.cuda.device_count() >= 1
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

OUTPUT_DIR = WORKING / "halluguard_minicheck_full_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Data and model


In [ ]:
raw = load_dataset(DATASET_ID, split=SPLIT, token=hf_token)
eval_df = raw.to_pandas()
eval_df["source_index"] = np.arange(len(eval_df), dtype=np.int64)
eval_df = eval_df.sort_values(["dataset", "source_index"], kind="stable").reset_index(drop=True)
dataset_names = sorted(eval_df["dataset"].unique())
assert len(dataset_names) == 11 and len(eval_df) == 29320
PILOT_SIZE, SUB_BATCH_SIZE = 100, 4
CHECKPOINT_PATH = OUTPUT_DIR / "predictions_checkpoint.csv"
FINAL_PATH, METRICS_PATH = OUTPUT_DIR / "predictions_final.csv", OUTPUT_DIR / "dataset_metrics.csv"
print(f"Loaded {len(eval_df):,} test rows across {len(dataset_names)} datasets.")


In [ ]:
scorer = MiniCheck(
    model_name="HalluGuard-Qwen3-4B",
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    max_model_len=MAX_MODEL_LEN,
    max_tokens=MAX_TOKENS,
    enable_prefix_caching=ENABLE_PREFIX_CACHING,
    cache_dir=CACHE_DIR,
)
assert scorer.model.operating_mode == "halluguard_binary"
assert scorer.model.model_id == MODEL_ID
assert scorer.model.model_revision == MODEL_REVISION
assert scorer.model.vllm_kwargs == {
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "gpu_memory_utilization": 0.80,
    "max_num_seqs": 16,
}
print("Loaded HalluGuard through MiniCheck on", TENSOR_PARALLEL_SIZE, "GPUs.")


## 4. Checkpointed pilot then full run


In [ ]:
RESULT_COLUMNS = ["source_index","dataset","label","prediction","support_score","status","error_type","error_message","raw_model_output","elapsed_seconds","attempt_count"]
def load_results():
    if not CHECKPOINT_PATH.exists(): return pd.DataFrame(columns=RESULT_COLUMNS)
    x = pd.read_csv(CHECKPOINT_PATH)
    return x.sort_values("attempt_count").drop_duplicates("source_index", keep="last")
def save_results(x):
    tmp = CHECKPOINT_PATH.with_suffix(".tmp"); x.to_csv(tmp,index=False); os.replace(tmp,CHECKPOINT_PATH)
def one_record(row, prior):
    started=time.perf_counter()
    try:
        p,s,_,_=scorer.score(docs=[str(row.doc)],claims=[str(row.claim)],chunk_size=CHUNK_SIZE)
        return {"source_index":int(row.source_index),"dataset":row.dataset,"label":int(row.label),"prediction":int(p[0]),"support_score":float(s[0]),"status":"OK","error_type":None,"error_message":None,"raw_model_output":None,"elapsed_seconds":time.perf_counter()-started,"attempt_count":prior+1}
    except Exception as exc:
        is_parse=isinstance(exc,HalluGuardParseError); message=str(exc)
        return {"source_index":int(row.source_index),"dataset":row.dataset,"label":int(row.label),"prediction":None,"support_score":None,"status":"PARSE_FAILURE" if is_parse else "INFERENCE_FAILURE","error_type":type(exc).__name__,"error_message":message[:8000],"raw_model_output":message[:8000] if is_parse else None,"elapsed_seconds":time.perf_counter()-started,"attempt_count":prior+1}
def counts(x):
    return {"total":len(x),"successful":int((x.status=="OK").sum()),"parse_failures":int((x.status=="PARSE_FAILURE").sum()),"inference_failures":int((x.status=="INFERENCE_FAILURE").sum())}
results=load_results(); done=set(results.source_index.astype(int)) if len(results) else set(); prior=dict(zip(results.source_index.astype(int),results.attempt_count.astype(int))) if len(results) else {}
pilot=eval_df.iloc[:PILOT_SIZE]; phases=[("pilot",pilot), ("full",eval_df)]
for name,frame in phases:
    for row in frame.itertuples(index=False):
        if int(row.source_index) in done: continue
        record=one_record(row,prior.get(int(row.source_index),0))
        results=pd.concat([results,pd.DataFrame([record])],ignore_index=True)
        done.add(int(row.source_index)); save_results(results)
        if len(results)%25==0: print(name,counts(results))
print("Actual checkpoint counts:",counts(results))


## 5. Metrics and artifacts

Metrics use successful rows only; parse and inference failures are reported separately.


In [ ]:
results=load_results()
merged=eval_df.merge(results,on=["source_index","dataset","label"],how="left",validate="one_to_one")
merged["status"]=merged["status"].fillna("NOT_RUN")
rows=[]
for name,g in merged.groupby("dataset",sort=True):
    ok=g[g.status=="OK"]; metrics={"accuracy":None,"f1":None,"balanced_accuracy":None}
    if len(ok):
        y=ok.label.astype(int); p=ok.prediction.astype(int)
        metrics={"accuracy":float(accuracy_score(y,p)),"f1":float(f1_score(y,p,zero_division=0)),"balanced_accuracy":float(balanced_accuracy_score(y,p)) if y.nunique()==2 else None}
    rows.append({"dataset":name,"total_samples":len(g),"successful_samples":len(ok),"parse_failures":int((g.status=="PARSE_FAILURE").sum()),"inference_failures":int((g.status=="INFERENCE_FAILURE").sum()),"parse_failure_rate":int((g.status=="PARSE_FAILURE").sum())/len(g),**metrics})
metrics=pd.DataFrame(rows); metrics.to_csv(METRICS_PATH,index=False); merged.to_csv(FINAL_PATH,index=False)
merged[merged.status=="PARSE_FAILURE"].to_csv(OUTPUT_DIR/"parse_failures_raw_output.csv",index=False)
overall=counts(merged); overall["parse_failure_rate"]=overall["parse_failures"]/len(merged); overall["macro_balanced_accuracy"]=float(metrics.balanced_accuracy.mean()) if metrics.balanced_accuracy.notna().any() else None
(OUTPUT_DIR/"manifest.json").write_text(json.dumps({"dataset":DATASET_ID,"split":SPLIT,"rows":len(eval_df),"chunk_size":CHUNK_SIZE,"max_model_len":MAX_MODEL_LEN,"max_tokens":MAX_TOKENS,"tensor_parallel_size":TENSOR_PARALLEL_SIZE,"generation":{"enable_thinking":False,"temperature":0,"format":"exact YES/NO"},"metric_handling":"metrics use successful rows only; failures have null predictions","counts":overall},indent=2),encoding="utf-8")
display(metrics); print("Actual full benchmark counts:",json.dumps(overall,indent=2))
